# Random Frame Pruning at 20% Retention on MELD

This notebook uniformly samples 20% of decoded candidate frames before Qwen2.5-VL-3B. The random subset is deterministic per sample ID, and selected frames are restored to chronological order. It extracts the random-frame representation, trains a new matched MLP, and compares it with the full baseline on identical test sample IDs.

## 1. Paths and experiment configuration

Before running, update the `benchmark-code` Kaggle dataset so it contains the new `--frame-selection random` implementation. Enable a GPU accelerator and internet access.

In [ ]:
from pathlib import Path

CODE = Path("/kaggle/input/datasets/pushkarsingh2005/benchmark-code")
INDEX_DIR = CODE / "data/indexes"

BASELINE_METRICS = CODE / "experiments/baselines/qwen2_5_vl_3b_shared/outputs/metrics.json"
BASELINE_PREDICTIONS = CODE / "experiments/baselines/qwen2_5_vl_3b_shared/outputs/test_predictions.csv"

RANDOM_ROOT = Path("/kaggle/working/embeddings/qwen2_5_vl_3b_random_frames_020_seed42")
RANDOM_CHUNKS = Path("/kaggle/working/embeddings/qwen2_5_vl_3b_random_frames_020_seed42_chunks")
RANDOM_RESULTS = Path("/kaggle/working/outputs/qwen2_5_vl_3b_random_frames_020_seed42")
SMOKE_OUTPUT = Path("/kaggle/working/smoke_random_frames_020_seed42.pt")

RETAIN_RATIO = 0.20
RANDOM_SEED = 42

for path in [RANDOM_ROOT, RANDOM_CHUNKS, RANDOM_RESULTS]:
    path.mkdir(parents=True, exist_ok=True)

extractor = CODE / "scripts/extraction/extract_qwen_shared_embeddings.py"
runner = CODE / "scripts/extraction/run_qwen_shared_chunks.py"
selector = CODE / "scripts/pruning/frame_diversity.py"

assert CODE.exists(), f"Missing Kaggle code dataset: {CODE}"
assert extractor.exists() and runner.exists() and selector.exists()
extractor_text = extractor.read_text()
selector_text = selector.read_text()
assert '--frame-random-seed' in extractor_text, 'Upload the updated benchmark-code dataset first.'
assert 'select_random_frame_indices' in selector_text
assert BASELINE_METRICS.exists() and BASELINE_PREDICTIONS.exists()
for split in ["train", "dev", "test"]:
    assert (INDEX_DIR / f"meld_{split}_index.csv").exists()

print("Code dataset:", CODE)
print("Random frame retain ratio:", RETAIN_RATIO)
print("Base random seed:", RANDOM_SEED)

## 2. Install dependencies and inspect the runtime

In [ ]:
!pip install -q -r {CODE}/requirements.txt
!pip install -q "transformers==5.0.0" "qwen-vl-utils[decord]==0.0.14"

In [ ]:
import json
import os

import numpy as np
import pandas as pd
import torch
import transformers

print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running extraction.")
print("GPU:", torch.cuda.get_device_name(0))
print("GPU memory (GiB):", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
print("HF token configured:", bool(os.environ.get("HF_TOKEN")))

for split in ["train", "dev", "test"]:
    index = pd.read_csv(INDEX_DIR / f"meld_{split}_index.csv")
    available = int(index["video_exists"].astype(bool).sum()) if "video_exists" in index else len(index)
    print(f"{split}: rows={len(index)}, available_videos={available}")

## 3. Two-sample smoke test

This verifies the new random-frame path and metadata before starting the full extraction. Only Qwen is loaded for visual processing.

In [ ]:
!python {extractor} \
  --index-csv {INDEX_DIR}/meld_dev_index.csv \
  --output-pt {SMOKE_OUTPUT} \
  --limit 2 \
  --fps 6 \
  --min-frames 4 \
  --max-frames 64 \
  --frame-size 224 \
  --pooling last \
  --prompt-style emotion_task \
  --modality-mode video_text \
  --frame-selection random \
  --frame-retain-ratio {RETAIN_RATIO} \
  --frame-random-seed {RANDOM_SEED} \
  --save-dtype float32 \
  --gc-every 1 \
  --batch-save-every 1

smoke = torch.load(SMOKE_OUTPUT, map_location="cpu", weights_only=False)
assert smoke["config"]["frame_selection"] == "random"
assert abs(float(smoke["config"]["frame_retain_ratio"]) - RETAIN_RATIO) < 1e-12
assert int(smoke["config"]["frame_random_seed"]) == RANDOM_SEED
for row in smoke["metadata"]:
    indices = row["selected_frame_indices"]
    expected = min(int(row["candidate_frame_count"]), max(1, round(int(row["candidate_frame_count"]) * RETAIN_RATIO)))
    assert indices == sorted(indices)
    assert len(indices) == len(set(indices)) == expected
    assert row["frame_random_sample_seed"] is not None
display(pd.DataFrame(smoke["metadata"])[[
    "sample_id", "candidate_frame_count", "retained_frame_count",
    "actual_frame_retain_ratio", "frame_random_sample_seed",
    "frame_selection_seconds", "video_token_count", "selected_frame_indices",
]])

## 4. Extract train, development, and test embeddings

Complete chunks with the same mode, ratio, and seed are skipped, so this cell is safe to rerun after a Kaggle interruption.

In [ ]:
for split in ["dev", "test", "train"]:
    !python {runner} \
      --index-csv {INDEX_DIR}/meld_{split}_index.csv \
      --chunks-dir {RANDOM_CHUNKS}/{split} \
      --chunk-prefix {split} \
      --chunk-size 500 \
      --fps 6 \
      --min-frames 4 \
      --max-frames 64 \
      --frame-size 224 \
      --pooling last \
      --prompt-style emotion_task \
      --modality-mode video_text \
      --frame-selection random \
      --frame-retain-ratio {RETAIN_RATIO} \
      --frame-random-seed {RANDOM_SEED} \
      --save-dtype float32 \
      --gc-every 5 \
      --batch-save-every 25 \
      --skip-existing-complete

## 5. Merge chunks and validate random selection

In [ ]:
for split in ["train", "dev", "test"]:
    !python {CODE}/scripts/extraction/merge_embedding_chunks.py \
      --chunks-dir {RANDOM_CHUNKS}/{split} \
      --output-pt {RANDOM_ROOT}/meld_{split}.pt \
      --pattern "{split}_*.pt"

In [ ]:
efficiency_rows = []

for split in ["train", "dev", "test"]:
    payload = torch.load(RANDOM_ROOT / f"meld_{split}.pt", map_location="cpu", weights_only=False)
    embeddings = payload["embeddings"].float()
    metadata = payload["metadata"]
    assert torch.isfinite(embeddings).all()
    assert embeddings.shape[0] == payload["labels"].shape[0] == len(payload["sample_ids"]) == len(metadata)

    for row in metadata:
        assert row["frame_selection"] == "random"
        assert row["frame_selection_implementation"] == "sample_id_seeded_uniform_random_frames"
        indices = row["selected_frame_indices"]
        expected = min(int(row["candidate_frame_count"]), max(1, round(int(row["candidate_frame_count"]) * RETAIN_RATIO)))
        assert indices == sorted(indices)
        assert len(indices) == len(set(indices)) == expected == int(row["retained_frame_count"])

    candidate = np.array([int(row["candidate_frame_count"]) for row in metadata])
    retained = np.array([int(row["retained_frame_count"]) for row in metadata])
    visual_tokens = np.array([int(row["video_token_count"]) for row in metadata])
    selection_seconds = np.array([float(row["frame_selection_seconds"]) for row in metadata])
    sample_seeds = [int(row["frame_random_sample_seed"]) for row in metadata]

    efficiency_rows.append({
        "split": split,
        "samples": len(payload["sample_ids"]),
        "errors": len(payload.get("errors", [])),
        "embedding_shape": str(tuple(embeddings.shape)),
        "mean_candidate_frames": candidate.mean(),
        "mean_retained_frames": retained.mean(),
        "actual_frame_ratio": retained.sum() / candidate.sum(),
        "mean_qwen_visual_tokens": visual_tokens.mean(),
        "mean_selection_seconds": selection_seconds.mean(),
        "p95_selection_seconds": np.percentile(selection_seconds, 95),
        "unique_sample_seeds": len(set(sample_seeds)),
    })

efficiency = pd.DataFrame(efficiency_rows)
display(efficiency.round(4))
efficiency.to_csv(RANDOM_RESULTS / "efficiency_summary.csv", index=False)

## 6. Train a new MLP on random-frame embeddings

The architecture and hyperparameters match the full baseline. Checkpoint selection uses development macro F1.

In [ ]:
NEW_MLP_RESULTS = RANDOM_RESULTS / "new_mlp"

!python {CODE}/scripts/classification/train_mlp.py \
  --train-pt {RANDOM_ROOT}/meld_train.pt \
  --dev-pt {RANDOM_ROOT}/meld_dev.pt \
  --test-pt {RANDOM_ROOT}/meld_test.pt \
  --output-dir {NEW_MLP_RESULTS} \
  --hidden-dim 512 \
  --dropout 0.3 \
  --epochs 50 \
  --batch-size 64 \
  --learning-rate 3e-4 \
  --weight-decay 1e-4 \
  --seed 42 \
  --device cuda

## 7. Native and exact sample-aligned comparison with the baseline

In [ ]:
baseline = json.loads(BASELINE_METRICS.read_text())
random_new = json.loads((NEW_MLP_RESULTS / "metrics.json").read_text())

def metric_row(representation, split, values):
    return {
        "Representation": representation,
        "Split": split,
        "Accuracy": values["accuracy"],
        "Macro F1": values["macro_f1"],
        "Weighted F1": values["weighted_f1"],
    }

native_results = pd.DataFrame([
    metric_row("Full baseline", "Dev", baseline["best_dev"]),
    metric_row("Full baseline", "Test", baseline["test"]),
    metric_row("Random frames 0.20 + new MLP", "Dev", random_new["best_dev"]),
    metric_row("Random frames 0.20 + new MLP", "Test", random_new["test"]),
])
display(native_results.round(4))
native_results.to_csv(RANDOM_RESULTS / "native_metrics_summary.csv", index=False)

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

prediction_paths = {
    "Full baseline": BASELINE_PREDICTIONS,
    "Random frames 0.20 + new MLP": NEW_MLP_RESULTS / "test_predictions.csv",
}
predictions = {name: pd.read_csv(path) for name, path in prediction_paths.items()}
for name, frame in predictions.items():
    assert frame["sample_id"].is_unique, f"Duplicate IDs in {name}"

common_ids = sorted(set.intersection(*(set(frame["sample_id"]) for frame in predictions.values())))
print("Exact shared test IDs:", len(common_ids))

aligned_rows = []
aligned_frames = {}
reference_truth = None
for name, frame in predictions.items():
    aligned = frame[frame["sample_id"].isin(common_ids)].sort_values("sample_id").reset_index(drop=True)
    assert len(aligned) == len(common_ids)
    truth = aligned[["sample_id", "true_id"]]
    if reference_truth is None:
        reference_truth = truth
    else:
        assert truth.equals(reference_truth), f"Ground-truth mismatch for {name}"
    aligned_frames[name] = aligned
    aligned_rows.append({
        "Representation": name,
        "n": len(aligned),
        "Accuracy": accuracy_score(aligned["true_id"], aligned["pred_id"]),
        "Macro F1": f1_score(aligned["true_id"], aligned["pred_id"], average="macro", zero_division=0),
        "Weighted F1": f1_score(aligned["true_id"], aligned["pred_id"], average="weighted", zero_division=0),
    })

aligned_results = pd.DataFrame(aligned_rows)
baseline_row = aligned_results.loc[aligned_results["Representation"] == "Full baseline"].iloc[0]
for metric in ["Accuracy", "Macro F1", "Weighted F1"]:
    aligned_results[f"Delta {metric}"] = aligned_results[metric] - baseline_row[metric]
display(aligned_results.round(4))

aligned_results.to_csv(RANDOM_RESULTS / "aligned_test_metrics.csv", index=False)
pd.DataFrame({"sample_id": common_ids}).to_csv(RANDOM_RESULTS / "aligned_test_sample_ids.csv", index=False)

## 8. Training curves, confusion matrices, and per-class metrics

In [ ]:
import matplotlib.pyplot as plt

history = pd.read_csv(NEW_MLP_RESULTS / "training_history.csv")
best_epoch = int(history.loc[history["dev_macro_f1"].idxmax(), "epoch"])
print("Random-frame MLP best epoch by dev macro F1:", best_epoch)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history["epoch"], history["train_loss"], label="train loss")
axes[0].plot(history["epoch"], history["dev_loss"], label="dev loss")
axes[0].axvline(best_epoch, color="black", linestyle="--", linewidth=1)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[1].plot(history["epoch"], history["dev_macro_f1"], label="dev macro F1")
axes[1].axvline(best_epoch, color="black", linestyle="--", linewidth=1)
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Macro F1")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
labels = random_new.get("label_names", ["anger", "disgust", "fear", "joy", "neutral", "sadness", "surprise"])

def labeled_confusion_matrix(split_metrics):
    return pd.DataFrame(
        split_metrics["confusion_matrix"],
        index=[f"true_{label}" for label in labels],
        columns=[f"pred_{label}" for label in labels],
    )

print("Random-frame development confusion matrix")
display(labeled_confusion_matrix(random_new["best_dev"]))
print("Random-frame test confusion matrix")
display(labeled_confusion_matrix(random_new["test"]))
print("Random-frame test classification report")
display(pd.DataFrame(random_new["test"]["classification_report"]).T.round(4))

## 9. Bundle result artifacts

In [ ]:
import shutil

archive_base = Path("/kaggle/working/qwen2_5_vl_3b_random_frames_020_seed42_results")
archive_path = shutil.make_archive(str(archive_base), "zip", root_dir=RANDOM_RESULTS)
print("Random embeddings directory:", RANDOM_ROOT)
print("Results directory:", RANDOM_RESULTS)
print("Downloadable results archive:", archive_path)
print("Result files:")
for path in sorted(RANDOM_RESULTS.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(RANDOM_RESULTS))